# 1.6 Tokenizer 训练 (Tokenizer Training)

> 🕐 预估学习时间：35分钟

分词器（Tokenizer）是 LLM 与文本之间的桥梁，决定了模型如何"看见"语言。本节从零实现主流分词算法，理解其原理与工程权衡。

本节涵盖：
- 分词器基础概念（字符级 / 词级 / 子词级）
- BPE (Byte Pair Encoding) 算法实现
- WordPiece 与 SentencePiece 对比
- 词表大小选择与多语言分词
- 特殊 Token 设计与聊天模板

## 1. 分词器基础概念

**为什么分词器很重要**：
- 分词器决定文本到 token 序列的映射，直接影响模型的"词汇"边界
- 词表大小影响 embedding 参数量和压缩率
- 好的分词器能平衡 OOV（未登录词）问题与序列长度

**三种分词粒度对比**：
| 粒度 | 优点 | 缺点 |
|------|------|------|
| 字符级 (Character) | 词表极小，无 OOV | 序列过长，语义稀疏 |
| 词级 (Word) | 序列短，语义明确 | 词表巨大，OOV 严重 |
| 子词级 (Subword) | 兼顾两者，OOV 少 | 算法复杂，需训练 |

**词表大小权衡**：
- 词表越大 → 压缩率越高（序列更短）→ 但 embedding 参数越多
- 词表越小 → 参数省 → 但序列更长，训练慢
- 主流 LLM 词表：32k~128k（GPT-2: 50k, LLaMA: 32k, Qwen: 152k）

In [ ]:
import torch
from collections import Counter

torch.manual_seed(42)


class CharTokenizer:
    # 字符级分词器：每个字符对应一个 token

    def __init__(self):
        self.char2id = {}
        self.id2char = {}
        self.vocab_size = 0

    def train(self, texts):
        special_tokens = ['[PAD]', '[UNK]', '[BOS]', '[EOS]']
        for i, tok in enumerate(special_tokens):
            self.char2id[tok] = i
            self.id2char[i] = tok
        chars = set()
        for text in texts:
            chars.update(text)
        for ch in sorted(chars):
            idx = len(self.char2id)
            self.char2id[ch] = idx
            self.id2char[idx] = ch
        self.vocab_size = len(self.char2id)

    def encode(self, text):
        unk_id = self.char2id['[UNK]']
        return [self.char2id.get(ch, unk_id) for ch in text]

    def decode(self, ids):
        return ''.join(self.id2char.get(i, '[UNK]') for i in ids)


class SimpleBPE:
    # 简化版 BPE：通过合并高频相邻 token 对来扩展词表

    def __init__(self, num_merges=50):
        self.num_merges = num_merges
        self.merges = []
        self.vocab = {}

    def _get_pairs(self, symbols):
        return [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]

    def _merge_symbols(self, symbols, pair):
        merged = pair[0] + pair[1]
        result = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                result.append(merged)
                i += 2
            else:
                result.append(symbols[i])
                i += 1
        return result

    def train(self, texts):
        word_freq = Counter()
        for text in texts:
            for word in text.split():
                symbols = tuple(list(word) + ['</w>'])
                word_freq[symbols] += 1
        vocab = set()
        for word in word_freq:
            vocab.update(word)
        self.vocab = {tok: i for i, tok in enumerate(sorted(vocab))}
        for step in range(self.num_merges):
            pair_counts = Counter()
            for word, freq in word_freq.items():
                for pair in self._get_pairs(list(word)):
                    pair_counts[pair] += freq
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            self.merges.append(best_pair)
            new_word_freq = Counter()
            for word, freq in word_freq.items():
                new_word = tuple(self._merge_symbols(list(word), best_pair))
                new_word_freq[new_word] += freq
            word_freq = new_word_freq
            merged = best_pair[0] + best_pair[1]
            if merged not in self.vocab:
                self.vocab[merged] = len(self.vocab)

    def encode(self, text):
        tokens = []
        for word in text.split():
            symbols = list(word) + ['</w>']
            for pair in self.merges:
                symbols = self._merge_symbols(symbols, pair)
            tokens.extend(symbols)
        return tokens


print('=== 分词器基础概念 ===')

zh_texts = ['自然语言处理很有趣', '语言模型需要分词器']
en_texts = ['natural language processing is fun', 'language models need tokenizers']

char_tok = CharTokenizer()
char_tok.train(zh_texts + en_texts)
print(f'\nCharTokenizer 词表大小: {char_tok.vocab_size}')
sample_zh = '语言模型'
ids = char_tok.encode(sample_zh)
print(f'编码 {sample_zh}: {ids}')
print(f'解码: {char_tok.decode(ids)}')

bpe = SimpleBPE(num_merges=30)
bpe.train(en_texts)
print(f'\nSimpleBPE 合并规则数: {len(bpe.merges)}')
print(f'前5条合并规则: {bpe.merges[:5]}')
sample_en = 'language models'
tokens = bpe.encode(sample_en)
print(f'编码 {sample_en}: {tokens}')

print(f'\nKey: 字符级分词器词表小但序列长，BPE 通过合并高频字节对平衡长度与词表。')

## 2. BPE (Byte Pair Encoding)

**算法核心思想**：从字符级开始，反复合并语料中出现频率最高的相邻 token 对。

**训练过程**：
1. 初始化：将所有词拆分为字符序列，末尾加 `</w>` 标记词边界
2. 统计所有相邻字符对的频率
3. 选出频率最高的对，合并为新 token，加入词表
4. 重复 2-3，直到达到目标词表大小或无更多合并

**停止条件**：
- 达到目标词表大小
- 没有更多可合并的对（频率低于阈值）
- 达到最大合并次数

**特殊 Token**：
- `[PAD]`：填充，用于对齐批次内序列长度
- `[UNK]`：未知 token，处理词表外的字符
- `[BOS]`：序列起始标记
- `[EOS]`：序列结束标记

**BPE 的优势**：
- 能处理任何语言和字符（包括未登录词）
- 词表大小可控，平衡压缩率与参数量
- 训练和推理都很快

In [ ]:
import torch
from collections import Counter

torch.manual_seed(42)


class BPETrainer:
    # 完整 BPE 训练器：支持词表大小控制和特殊 token

    SPECIAL_TOKENS = ['[PAD]', '[UNK]', '[BOS]', '[EOS]']

    def __init__(self, vocab_size=500):
        self.target_vocab_size = vocab_size
        self.merges = []
        self.token2id = {}
        self.id2token = {}

    def train(self, corpus):
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            self.token2id[tok] = i
            self.id2token[i] = tok
        char_set = set()
        for text in corpus:
            char_set.update(text)
        for ch in sorted(char_set):
            idx = len(self.token2id)
            self.token2id[ch] = idx
            self.id2token[idx] = ch
        word_freq = Counter()
        for text in corpus:
            for word in text.split():
                word_freq[tuple(list(word) + ['</w>'])] += 1
        while len(self.token2id) < self.target_vocab_size:
            pair_counts = Counter()
            for word, freq in word_freq.items():
                for i in range(len(word) - 1):
                    pair_counts[(word[i], word[i + 1])] += freq
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            self.merges.append(best_pair)
            merged = best_pair[0] + best_pair[1]
            idx = len(self.token2id)
            self.token2id[merged] = idx
            self.id2token[idx] = merged
            new_word_freq = Counter()
            for word, freq in word_freq.items():
                new_word = self._apply_merge(word, best_pair)
                new_word_freq[new_word] += freq
            word_freq = new_word_freq

    def _apply_merge(self, word, pair):
        result = []
        i = 0
        merged = pair[0] + pair[1]
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                result.append(merged)
                i += 2
            else:
                result.append(word[i])
                i += 1
        return tuple(result)

    def encode(self, text):
        ids = [self.token2id['[BOS]']]
        unk_id = self.token2id['[UNK]']
        for word in text.split():
            symbols = list(word) + ['</w>']
            for pair in self.merges:
                symbols = list(self._apply_merge(tuple(symbols), pair))
            for sym in symbols:
                ids.append(self.token2id.get(sym, unk_id))
        ids.append(self.token2id['[EOS]'])
        return ids

    def decode(self, ids):
        tokens = [self.id2token.get(i, '[UNK]') for i in ids]
        text = ''.join(tokens)
        text = text.replace('</w>', ' ')
        text = text.replace('[BOS]', '').replace('[EOS]', '')
        return text.strip()


corpus = [
    'the quick brown fox jumps over the lazy dog',
    'the dog and the fox are friends',
    'quick brown dogs run fast',
    'lazy foxes sleep all day',
    'the quick fox and the lazy dog',
]

print('=== BPE 训练器 ===')
trainer = BPETrainer(vocab_size=80)
trainer.train(corpus)
print(f'\n目标词表大小: {trainer.target_vocab_size}')
print(f'实际词表大小: {len(trainer.token2id)}')
print(f'合并规则数: {len(trainer.merges)}')
print(f'\n前10条合并规则:')
for i, (a, b) in enumerate(trainer.merges[:10]):
    merged = a + b
    print(f'  {i+1}. {a} + {b} -> {merged}')

sample = 'the quick fox'
print(f'\n编码前: {sample}')
ids = trainer.encode(sample)
print(f'编码后: {ids}')
tokens = [trainer.id2token[i] for i in ids]
print(f'Token 序列: {tokens}')
print(f'解码后: {trainer.decode(ids)}')

print(f'\nKey: BPE 通过贪心合并最高频字节对扩展词表，直到达到目标大小。')

## 3. WordPiece 与 SentencePiece

**WordPiece 与 BPE 的区别**：
- BPE 选频率最高的对合并
- WordPiece 选"似然增益"最高的对合并
- 似然评分 = pair_freq / (freq_a × freq_b)
- 这使得 WordPiece 倾向合并"各自不常见但合在一起常见"的对

**SentencePiece 的特点**：
- 语言无关：直接处理原始字节/字符，不依赖空格分词
- 支持多种子词算法（BPE, Unigram, Char）
- 将空格转为特殊符号 `▁`，实现可逆解码
- 适用于中日韩等无空格语言

**Unigram 语言模型**（SentencePiece 选项之一）：
- 假设词表中的 token 独立产生，目标是最大化语料似然
- 训练时用 EM 算法迭代优化
- 推理时用 Viterbi 找最大似然切分

**主流模型使用的分词器**：
| 模型 | 分词器 | 词表大小 |
|------|--------|----------|
| BERT | WordPiece | 30k |
| GPT-2/3 | BPE | 50k |
| LLaMA | SentencePiece (BPE) | 32k |
| T5 | SentencePiece (Unigram) | 32k |
| Qwen | tiktoken (BPE) | 152k |

In [ ]:
import torch
from collections import Counter
import math

torch.manual_seed(42)


class WordPieceTrainer:
    # WordPiece 训练器：基于似然的合并评分

    def __init__(self, vocab_size=100):
        self.target_vocab_size = vocab_size
        self.merges = []
        self.vocab = {}

    def train(self, corpus):
        word_freq = Counter()
        for text in corpus:
            for word in text.split():
                word_freq[tuple(list(word) + ['##end'])] += 1
        vocab = set()
        for word in word_freq:
            vocab.update(word)
        self.vocab = {tok: i for i, tok in enumerate(sorted(vocab))}
        while len(self.vocab) < self.target_vocab_size:
            pair_counts = Counter()
            char_freq = Counter()
            for word, freq in word_freq.items():
                for sym in word:
                    char_freq[sym] += freq
                for i in range(len(word) - 1):
                    pair_counts[(word[i], word[i + 1])] += freq
            if not pair_counts:
                break
            best_pair = None
            best_score = -1.0
            for pair, count in pair_counts.items():
                fa = char_freq[pair[0]]
                fb = char_freq[pair[1]]
                if fa == 0 or fb == 0:
                    continue
                score = math.log(count) - math.log(fa) - math.log(fb)
                if score > best_score:
                    best_score = score
                    best_pair = pair
            if best_pair is None:
                break
            self.merges.append(best_pair)
            merged = best_pair[0] + best_pair[1]
            if merged not in self.vocab:
                self.vocab[merged] = len(self.vocab)
            new_word_freq = Counter()
            for word, freq in word_freq.items():
                new_word = self._merge(word, best_pair)
                new_word_freq[new_word] += freq
            word_freq = new_word_freq

    def _merge(self, word, pair):
        result = []
        i = 0
        merged = pair[0] + pair[1]
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                result.append(merged)
                i += 2
            else:
                result.append(word[i])
                i += 1
        return tuple(result)

    def encode(self, text):
        tokens = []
        for word in text.split():
            symbols = list(word) + ['##end']
            for pair in self.merges:
                symbols = list(self._merge(tuple(symbols), pair))
            tokens.extend(symbols)
        return tokens


class BPETokenizer:
    # BPE 分词器（用于对比）

    def __init__(self, vocab_size=100):
        self.target_vocab_size = vocab_size
        self.merges = []
        self.vocab = {}

    def train(self, corpus):
        word_freq = Counter()
        for text in corpus:
            for word in text.split():
                word_freq[tuple(list(word) + ['</w>'])] += 1
        vocab = set()
        for word in word_freq:
            vocab.update(word)
        self.vocab = {tok: i for i, tok in enumerate(sorted(vocab))}
        while len(self.vocab) < self.target_vocab_size:
            pair_counts = Counter()
            for word, freq in word_freq.items():
                for i in range(len(word) - 1):
                    pair_counts[(word[i], word[i + 1])] += freq
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            self.merges.append(best_pair)
            merged = best_pair[0] + best_pair[1]
            if merged not in self.vocab:
                self.vocab[merged] = len(self.vocab)
            new_word_freq = Counter()
            for word, freq in word_freq.items():
                new_word = self._merge(word, best_pair)
                new_word_freq[new_word] += freq
            word_freq = new_word_freq

    def _merge(self, word, pair):
        result = []
        i = 0
        merged = pair[0] + pair[1]
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                result.append(merged)
                i += 2
            else:
                result.append(word[i])
                i += 1
        return tuple(result)

    def encode(self, text):
        tokens = []
        for word in text.split():
            symbols = list(word) + ['</w>']
            for pair in self.merges:
                symbols = list(self._merge(tuple(symbols), pair))
            tokens.extend(symbols)
        return tokens


corpus = [
    'the quick brown fox jumps',
    'the lazy dog sleeps',
    'quick foxes and lazy dogs',
    'brown dogs run fast',
    'the fox and the dog',
]

print('=== WordPiece vs BPE 对比 ===')
wp = WordPieceTrainer(vocab_size=60)
wp.train(corpus)
bpe = BPETokenizer(vocab_size=60)
bpe.train(corpus)

sample = 'the quick fox'
wp_tokens = wp.encode(sample)
bpe_tokens = bpe.encode(sample)

print(f'\n样本: {sample}')
print(f'WordPiece: {wp_tokens}')
print(f'BPE:       {bpe_tokens}')

print(f'\n%-15s %-12s %-10s %-10s' % ('算法', '词表大小', '合并数', 'token数'))
print('-' * 50)
print('%-15s %-12d %-10d %-10d' % ('WordPiece', len(wp.vocab), len(wp.merges), len(wp_tokens)))
print('%-15s %-12d %-10d %-10d' % ('BPE', len(bpe.vocab), len(bpe.merges), len(bpe_tokens)))

print(f'\nKey: BPE 选最高频对，WordPiece 选似然最高的对（频率除以各部分频率乘积）。')

## 4. 词表大小选择与多语言分词

**词表大小的核心权衡**：
- **大词表**：压缩率高（序列短）→ 训练快、推理快；但 embedding 参数多
- **小词表**：参数省；但序列长 → 训练慢、上下文窗口浪费

**参数量估算**：
- embedding 参数 = vocab_size × hidden_size
- 例如：32k × 4096 = 1.3 亿参数（LLaMA-7B 的 embedding 占比）

**多语言分词的挑战**：
- 中文/日文等无空格语言需要特殊处理
- 字节级（byte-level）BPE 可处理任意字符，无 OOV
- 多语言模型需要平衡各语言的压缩率（避免某语言被"惩罚"）

**字节级 vs 字符级**：
| 方式 | 词表基座 | 优点 | 缺点 |
|------|----------|------|------|
| 字符级 | Unicode 字符 | 直观 | 罕见字符仍 OOV |
| 字节级 | 256 字节 | 永不 OOV | 序列更长（中文 3x） |

**产业实践**：
- GPT-2 起广泛采用 byte-level BPE
- 多语言模型（如 mBART, XLM-R）通常用 250k+ 词表
- 中文模型常用 150k+ 词表以提升压缩率

In [ ]:
import torch
import numpy as np
from collections import Counter
import random

torch.manual_seed(42)
random.seed(42)


class BPEForAnalysis:
    def __init__(self, vocab_size=1000):
        self.target_vocab_size = vocab_size
        self.merges = []

    def train(self, corpus):
        word_freq = Counter()
        for text in corpus:
            for word in text.split():
                word_freq[tuple(list(word) + ['</w>'])] += 1
        vocab = set()
        for word in word_freq:
            vocab.update(word)
        current_size = len(vocab)
        while current_size < self.target_vocab_size:
            pair_counts = Counter()
            for word, freq in word_freq.items():
                for i in range(len(word) - 1):
                    pair_counts[(word[i], word[i + 1])] += freq
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            self.merges.append(best_pair)
            current_size += 1
            new_word_freq = Counter()
            for word, freq in word_freq.items():
                new_word = self._merge(word, best_pair)
                new_word_freq[new_word] += freq
            word_freq = new_word_freq

    def _merge(self, word, pair):
        result = []
        i = 0
        merged = pair[0] + pair[1]
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                result.append(merged)
                i += 2
            else:
                result.append(word[i])
                i += 1
        return tuple(result)

    def encode(self, text):
        tokens = []
        for word in text.split():
            symbols = list(word) + ['</w>']
            for pair in self.merges:
                symbols = list(self._merge(tuple(symbols), pair))
            tokens.extend(symbols)
        return tokens


base_words = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'lazy', 'dog',
              'language', 'model', 'tokenizer', 'training', 'data', 'learning',
              'natural', 'processing', 'neural', 'network', 'transformer', 'attention']
corpus = []
for _ in range(500):
    n_words = random.randint(5, 15)
    corpus.append(' '.join(random.choices(base_words, k=n_words)))

eval_text = 'the quick brown fox jumps over the lazy dog language model training'
n_chars = len(eval_text.replace(' ', ''))

print('=== 词表大小分析 ===')
print('\n%-12s %-15s %-12s %-12s' % ('词表大小', '合并规则数', 'token数', '压缩率'))
print('-' * 55)

compression_ratios = []
for vs in [50, 200, 500, 1000, 2000]:
    bpe = BPEForAnalysis(vocab_size=vs)
    bpe.train(corpus)
    tokens = bpe.encode(eval_text)
    compression = len(tokens) / n_chars
    compression_ratios.append(compression)
    print('%-12d %-15d %-12d %-12.3f' % (vs, len(bpe.merges), len(tokens), compression))

comp_arr = np.array(compression_ratios)
print(f'\n压缩率范围: {comp_arr.min():.3f} ~ {comp_arr.max():.3f}')
print(f'压缩率下降: {(1 - comp_arr[-1] / comp_arr[0]) * 100:.1f}%（从最小词表到最大词表）')

print(f'\nKey: 词表越大，压缩率越高（token 数越少），但 embedding 参数线性增长。')

## 5. 特殊 Token 设计与实践

**聊天模板中的特殊 Token**：
- `[USER]` / `[ASSISTANT]` / `[SYSTEM]`：标记对话角色
- `[TOOL]`：标记工具调用结果
- 这些 token 让模型学会区分对话结构

**Function Calling 的 Token 设计**：
```
[USER] 帮我查一下北京天气 [TOOL_CALL] get_weather(city='北京') [/TOOL_CALL]
[TOOL] {'temp': 25, 'weather': '晴'} [/TOOL]
[ASSISTANT] 北京今天 25 度，晴天。
```

**Padding 策略**：
- **Right Padding**：`[PAD]` 加在序列末尾（训练常用）
- **Left Padding**：`[PAD]` 加在序列开头（生成任务常用，保持最后 token 位置）
- **Batch 动态填充**：按批次内最长序列填充，减少浪费

**特殊 Token 训练注意事项**：
- 新增特殊 token 后，对应的 embedding 行需要初始化（通常随机初始化 + 微调）
- 注意调整 `model.resize_token_embeddings(len(tokenizer))`
- 特殊 token 的 ID 应固定，避免版本间不一致

In [ ]:
import torch
from collections import Counter

torch.manual_seed(42)


class ChatBPE:
    # 带聊天特殊 token 的 BPE 分词器

    CHAT_TOKENS = ['[USER]', '[ASSISTANT]', '[SYSTEM]', '[TOOL]',
                   '[PAD]', '[UNK]', '[BOS]', '[EOS]']

    def __init__(self, vocab_size=200):
        self.target_vocab_size = vocab_size
        self.merges = []
        self.token2id = {}
        self.id2token = {}

    def train(self, corpus):
        for i, tok in enumerate(self.CHAT_TOKENS):
            self.token2id[tok] = i
            self.id2token[i] = tok
        char_set = set()
        for text in corpus:
            char_set.update(text)
        for ch in sorted(char_set):
            idx = len(self.token2id)
            self.token2id[ch] = idx
            self.id2token[idx] = ch
        word_freq = Counter()
        for text in corpus:
            for word in text.split():
                word_freq[tuple(list(word) + ['</w>'])] += 1
        while len(self.token2id) < self.target_vocab_size:
            pair_counts = Counter()
            for word, freq in word_freq.items():
                for i in range(len(word) - 1):
                    pair_counts[(word[i], word[i + 1])] += freq
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            self.merges.append(best_pair)
            merged = best_pair[0] + best_pair[1]
            idx = len(self.token2id)
            self.token2id[merged] = idx
            self.id2token[idx] = merged
            new_word_freq = Counter()
            for word, freq in word_freq.items():
                new_word = self._merge(word, best_pair)
                new_word_freq[new_word] += freq
            word_freq = new_word_freq

    def _merge(self, word, pair):
        result = []
        i = 0
        merged = pair[0] + pair[1]
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                result.append(merged)
                i += 2
            else:
                result.append(word[i])
                i += 1
        return tuple(result)

    def _encode_word(self, word):
        symbols = list(word) + ['</w>']
        for pair in self.merges:
            symbols = list(self._merge(tuple(symbols), pair))
        return symbols

    def encode_chat(self, messages):
        ids = [self.token2id['[BOS]']]
        unk_id = self.token2id['[UNK]']
        for msg in messages:
            role = msg['role']
            content = msg['content']
            role_token = '[' + role.upper() + ']'
            ids.append(self.token2id[role_token])
            for word in content.split():
                symbols = self._encode_word(word)
                for sym in symbols:
                    ids.append(self.token2id.get(sym, unk_id))
        ids.append(self.token2id['[EOS]'])
        return ids

    def decode_chat(self, ids):
        return [self.id2token.get(i, '[UNK]') for i in ids]


corpus = [
    'hello how are you today',
    'i am fine thank you',
    'the weather is nice today',
    'can you help me with this task',
    'sure i can help you with that',
    'what tools do you have available',
    'i can search the web and run code',
]

print('=== 聊天模板分词器 ===')
chat_tok = ChatBPE(vocab_size=150)
chat_tok.train(corpus)

conversation = [
    {'role': 'system', 'content': 'you are a helpful assistant'},
    {'role': 'user', 'content': 'hello how are you'},
    {'role': 'assistant', 'content': 'i am fine thank you'},
    {'role': 'user', 'content': 'can you help me'},
    {'role': 'assistant', 'content': 'sure i can help you'},
]

ids = chat_tok.encode_chat(conversation)
tokens = chat_tok.decode_chat(ids)

print(f'\n对话轮数: {len(conversation)}')
print(f'Token 总数: {len(ids)}')
print(f'\nToken 序列:')
for i, tok in enumerate(tokens):
    marker = ' <<<' if tok in chat_tok.CHAT_TOKENS else ''
    print(f'  {i:3d}: {tok}{marker}')

print(f'\n特殊 token: {chat_tok.CHAT_TOKENS}')

print(f'\nKey: 聊天模板用特殊 token 标记角色边界，让模型学会区分对话结构。')

## 📝 课后思考题

1. BPE 的合并顺序对最终分词结果有什么影响？如果改变合并顺序会怎样？
2. 为什么 WordPiece 用似然评分而不是频率？在什么场景下两者结果差异最大？
3. 字节级 BPE 相比字符级 BPE 有什么优势？为什么 GPT-2 选择字节级？
4. 设计一个支持多轮对话和工具调用的聊天模板，需要哪些特殊 token？如何组织 token 序列？